In [2]:
import pandas as pd

df = pd.read_csv("synthetic_audit_log.csv")

print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nLabel balance:")
print(df["label"].value_counts(normalize=True))

Shape: (15936, 11)

Columns: ['auditID', 'actorID', 'actorRole', 'actorOrg', 'action', 'resourceType', 'resourceID', 'patientID', 'timestamp', 'label', 'anomaly_type']

Label balance:
label
normal       0.928589
anomalous    0.071411
Name: proportion, dtype: float64


In [3]:
from feature_engineering import build_actor_features

actor_features = build_actor_features(df)

print("Shape:", actor_features.shape)
print("\nColumns:", list(actor_features.columns))
actor_features.head()

Shape: (4704, 12)

Columns: ['actorID', 'actorRole', 'actorOrg', 'date', 'access_frequency', 'distinct_resources_touched', 'off_hours_ratio', 'burst_count', 'access_denied_ratio', 'consent_denied_ratio', 'dispense_without_read_count', 'label']


,actorID,actorRole,actorOrg,date,access_frequency,distinct_resources_touched,off_hours_ratio,burst_count,access_denied_ratio,consent_denied_ratio,dispense_without_read_count,label
0,doctor-001,doctor,HospitalMSP,2026-01-01,3,3,0.000000,0,0.0,0.0,0,normal
1,doctor-002,doctor,,2026-01-01,9,9,0.111111,0,0.0,0.0,0,normal
2,doctor-003,doctor,,2026-01-01,5,5,0.200000,0,0.0,0.0,0,normal
3,doctor-004,doctor,HospitalMSP,2026-01-01,3,3,0.000000,0,0.0,0.0,0,normal
4,doctor-005,doctor,,2026-01-01,10,10,0.100000,0,0.0,0.0,0,normal


In [4]:
from pycaret.anomaly import setup

exp = setup(
    data=actor_features.drop(columns=["label"]),
    ignore_features=["actorID", "actorRole", "actorOrg", "date"],
    session_id=42,
)

,Description,Value
0,Session id,42
1,Original data shape,"(4704, 11)"
2,Transformed data shape,"(4704, 7)"
3,Ignore features,4
4,Numeric features,7
5,Preprocess,True
6,Imputation type,simple
7,Numeric imputation,mean
8,Categorical imputation,mode
9,CPU Jobs,-1


In [4]:
from pycaret.anomaly import create_model, assign_model
from sklearn.metrics import average_precision_score

algos = ["iforest", "knn", "lof", "cluster", "svm", "ecod"]
results = {}

for algo in algos:
    model = create_model(algo, fraction=0.0714)
    scored = assign_model(model)
    ap = average_precision_score(actor_features["label"] == "anomalous", scored["Anomaly_Score"])
    results[algo] = ap
    print(f"{algo:10s} -> Average Precision: {ap:.4f}")

ValueError: _CURRENT_EXPERIMENT global variable is not set. Please run setup() first.

In [6]:
from pycaret.anomaly import models

models()

,Name,Reference
ID,,
abod,Angle-base Outlier Detection,pyod.models.abod.ABOD
cluster,Clustering-Based Local Outlier,pycaret.internal.patches.pyod.CBLOFForceToDouble
cof,Connectivity-Based Local Outlier,pyod.models.cof.COF
iforest,Isolation Forest,pyod.models.iforest.IForest
histogram,Histogram-based Outlier Detection,pyod.models.hbos.HBOS
knn,K-Nearest Neighbors Detector,pyod.models.knn.KNN
lof,Local Outlier Factor,pyod.models.lof.LOF
svm,One-class SVM detector,pyod.models.ocsvm.OCSVM
pca,Principal Component Analysis,pyod.models.pca.PCA


In [ ]:
more_algos = ["histogram", "pca", "mcd"]

for algo in more_algos:
    model = create_model(algo, fraction=0.0714)
    scored = assign_model(model)
    ap = average_precision_score(actor_features["label"] == "anomalous", scored["Anomaly_Score"])
    results[algo] = ap
    print(f"{algo:10s} -> Average Precision: {ap:.4f}")

In [5]:
FEATURE_COLUMNS = [
    "access_frequency",
    "distinct_resources_touched",
    "off_hours_ratio",
    "burst_count",
    "access_denied_ratio",
    "consent_denied_ratio",
    "dispense_without_read_count",
]

def split_train_holdout(df, train_frac: float = 0.8):
    df = df.sort_values(["actorID", "date"]).reset_index(drop=True)
    df["held_out"] = False
    for actor_id, group in df.groupby("actorID"):
        cutoff = max(1, int(len(group) * train_frac))
        holdout_idx = group.index[cutoff:]
        df.loc[holdout_idx, "held_out"] = True
    return df

split_df = split_train_holdout(actor_features, train_frac=0.8)
print(split_df["held_out"].value_counts())

held_out
False    3645
True     1059
Name: count, dtype: int64


In [6]:
print("Train label balance:")
print(split_df[~split_df["held_out"]]["label"].value_counts())

print("\nHeld-out label balance:")
print(split_df[split_df["held_out"]]["label"].value_counts())

Train label balance:
label
normal       3579
anomalous      66
Name: count, dtype: int64

Held-out label balance:
label
normal       1044
anomalous      15
Name: count, dtype: int64


In [7]:
split_df = split_train_holdout(actor_features, train_frac=0.6)

print("Train label balance:")
print(split_df[~split_df["held_out"]]["label"].value_counts())

print("\nHeld-out label balance:")
print(split_df[split_df["held_out"]]["label"].value_counts())

Train label balance:
label
normal       2658
anomalous      46
Name: count, dtype: int64

Held-out label balance:
label
normal       1965
anomalous      35
Name: count, dtype: int64


In [8]:
from pyod.models.pca import PCA
from pyod.models.knn import KNN
from pyod.models.iforest import IForest
from pyod.models.ocsvm import OCSVM
from pyod.models.hbos import HBOS

train_df = split_df[~split_df["held_out"]]
X_train = train_df[FEATURE_COLUMNS].values
X_all = split_df[FEATURE_COLUMNS].values

contamination = 0.0714  # matches the actual label rate in the raw data

In [9]:
#fit first model
pca_model = PCA(contamination=contamination, random_state=42)
pca_model.fit(X_train)

pca_scores = pca_model.decision_function(X_all)
split_df["pca_score"] = pca_scores

print("PCA fitted. Score range:", pca_scores.min(), "to", pca_scores.max())

PCA fitted. Score range: 4.287775173979953e+29 to 1.1842263954252868e+31


In [10]:
#adding standardisation before fitting
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_all_scaled = scaler.transform(X_all)

pca_model = PCA(contamination=contamination, random_state=42)
pca_model.fit(X_train_scaled)

pca_scores = pca_model.decision_function(X_all_scaled)
split_df["pca_score"] = pca_scores

print("PCA fitted. Score range:", pca_scores.min(), "to", pca_scores.max())

PCA fitted. Score range: 4.579120517312496e+29 to 1.2646920989100123e+31


In [11]:
import numpy as np
import pandas as pd

variances = pd.Series(X_train_scaled.var(axis=0), index=FEATURE_COLUMNS)
print(variances.sort_values())

off_hours_ratio                1.0
access_frequency               1.0
distinct_resources_touched     1.0
burst_count                    1.0
access_denied_ratio            1.0
consent_denied_ratio           1.0
dispense_without_read_count    1.0
dtype: float64


In [15]:
corr = pd.DataFrame(X_train_scaled, columns=FEATURE_COLUMNS).corr()
print("Correlation matrix:")
print(corr.round(2))

print("\nExplained variance ratio per component:")
print(pca_model.explained_variance_ratio_)

Correlation matrix:
                             access_frequency  distinct_resources_touched  \
access_frequency                         1.00                        0.95   
distinct_resources_touched               0.95                        1.00   
off_hours_ratio                          0.06                        0.07   
burst_count                              0.56                        0.47   
access_denied_ratio                      0.22                        0.10   
consent_denied_ratio                     0.22                        0.10   
dispense_without_read_count              0.14                        0.12   

                             off_hours_ratio  burst_count  \
access_frequency                        0.06         0.56   
distinct_resources_touched              0.07         0.47   
off_hours_ratio                         1.00         0.09   
burst_count                             0.09         1.00   
access_denied_ratio                    -0.03         0.43 

In [12]:
#access_denied_ratio and consent_denied_ratio are perfectly correlated (1.00)
FEATURE_COLUMNS = [c for c in FEATURE_COLUMNS if c != "access_denied_ratio"]
print("Using features:", FEATURE_COLUMNS)

X_train = train_df[FEATURE_COLUMNS].values
X_all = split_df[FEATURE_COLUMNS].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_all_scaled = scaler.transform(X_all)

Using features: ['access_frequency', 'distinct_resources_touched', 'off_hours_ratio', 'burst_count', 'consent_denied_ratio', 'dispense_without_read_count']


In [13]:
pca_model = PCA(contamination=contamination, random_state=42)
pca_model.fit(X_train_scaled)

pca_scores = pca_model.decision_function(X_all_scaled)
split_df["pca_score"] = pca_scores

print("PCA fitted. Score range:", pca_scores.min(), "to", pca_scores.max())
print("\nExplained variance ratio per component:")
print(pca_model.explained_variance_ratio_)

PCA fitted. Score range: 96.38269750197091 to 5405.322163524755

Explained variance ratio per component:
[0.41581364 0.17994774 0.17040485 0.15346109 0.07374869 0.00662398]


In [14]:
holdout_mask = split_df["held_out"]
y_holdout = (split_df.loc[holdout_mask, "label"] == "anomalous").astype(int)

pca_ap = average_precision_score(y_holdout, split_df.loc[holdout_mask, "pca_score"])
print(f"PCA holdout Average Precision: {pca_ap:.4f}")

PCA holdout Average Precision: 0.7866


In [15]:
knn_model = KNN(contamination=contamination)
knn_model.fit(X_train_scaled)

knn_scores = knn_model.decision_function(X_all_scaled)
split_df["knn_score"] = knn_scores

knn_ap = average_precision_score(y_holdout, split_df.loc[holdout_mask, "knn_score"])
print(f"KNN holdout Average Precision: {knn_ap:.4f}")

KNN holdout Average Precision: 0.7863


In [16]:
iforest_model = IForest(contamination=contamination, random_state=42)
iforest_model.fit(X_train_scaled)

iforest_scores = iforest_model.decision_function(X_all_scaled)
split_df["iforest_score"] = iforest_scores

iforest_ap = average_precision_score(y_holdout, split_df.loc[holdout_mask, "iforest_score"])
print(f"IForest holdout Average Precision: {iforest_ap:.4f}")

IForest holdout Average Precision: 0.7423


In [17]:
ocsvm_model = OCSVM(contamination=contamination)
ocsvm_model.fit(X_train_scaled)

ocsvm_scores = ocsvm_model.decision_function(X_all_scaled)
split_df["ocsvm_score"] = ocsvm_scores

ocsvm_ap = average_precision_score(y_holdout, split_df.loc[holdout_mask, "ocsvm_score"])
print(f"OCSVM holdout Average Precision: {ocsvm_ap:.4f}")

OCSVM holdout Average Precision: 0.7899


In [18]:
hbos_model = HBOS(contamination=contamination)
hbos_model.fit(X_train_scaled)

hbos_scores = hbos_model.decision_function(X_all_scaled)
split_df["hbos_score"] = hbos_scores

hbos_ap = average_precision_score(y_holdout, split_df.loc[holdout_mask, "hbos_score"])
print(f"HBOS holdout Average Precision: {hbos_ap:.4f}")

HBOS holdout Average Precision: 0.6517


In [19]:
models_fitted = {
    "pca": pca_model,
    "knn": knn_model,
    "iforest": iforest_model,
    "ocsvm": ocsvm_model,
    "hbos": hbos_model,
}

holdout_df = split_df[holdout_mask].copy()

for name, model in models_fitted.items():
    holdout_df[f"{name}_pred"] = model.predict(X_all_scaled[holdout_mask.values])

pred_cols = [f"{name}_pred" for name in models_fitted]
holdout_df["votes"] = holdout_df[pred_cols].sum(axis=1)

# Only look at the actual anomalous rows
true_anomalies = holdout_df[holdout_df["label"] == "anomalous"]
print(true_anomalies[["actorID", "date", "votes"] + pred_cols])

                 actorID        date  votes  pca_pred  knn_pred  iforest_pred  \
351           doctor-007  2026-02-26      5         1         1             1   
508           doctor-010  2026-02-24      5         1         1             1   
524           doctor-010  2026-03-23      3         0         1             1   
565           doctor-011  2026-03-06      5         1         1             1   
662           doctor-013  2026-02-23      1         0         0             0   
728           doctor-014  2026-03-21      5         1         1             1   
787           doctor-015  2026-03-28      5         1         1             1   
898           doctor-017  2026-03-21      5         1         1             1   
943           doctor-018  2026-02-23      5         1         1             1   
1007          doctor-019  2026-03-13      5         1         1             1   
1166          doctor-022  2026-03-08      0         0         0             0   
1169          doctor-022  20

In [20]:
df["date"] = pd.to_datetime(df["timestamp"]).dt.date

anomaly_type_map = (
    df[df["label"] == "anomalous"]
    .groupby(["actorID", "date"])["anomaly_type"]
    .first()
)

holdout_df["anomaly_type"] = holdout_df.set_index(["actorID", "date"]).index.map(anomaly_type_map)

# Zero-vote misses
misses = holdout_df[(holdout_df["label"] == "anomalous") & (holdout_df["votes"] == 0)]
print("Zero-vote misses (anomaly_type):")
print(misses[["actorID", "date", "anomaly_type"]])

# False positives: normal rows with 3+ votes
false_positives = holdout_df[(holdout_df["label"] == "normal") & (holdout_df["votes"] >= 3)]
print(f"\nFalse positives (normal rows with 3+ votes): {len(false_positives)}")
print(false_positives[["actorID", "date", "votes"]])

Zero-vote misses (anomaly_type):
                 actorID        date       anomaly_type
1166          doctor-022  2026-03-08  order_composition
1309          doctor-025  2026-03-06  order_composition
2038  orgactor-pharm-003  2026-03-14   bigram_violation
2103  orgactor-pharm-004  2026-03-07   bigram_violation
2168  orgactor-pharm-005  2026-03-05   bigram_violation

False positives (normal rows with 3+ votes): 41
                 actorID        date  votes
38            doctor-001  2026-03-12      3
94            doctor-002  2026-03-15      4
154           doctor-003  2026-03-23      5
201           doctor-004  2026-03-12      4
202           doctor-004  2026-03-13      3
260           doctor-005  2026-03-30      4
307           doctor-006  2026-03-20      4
368           doctor-007  2026-03-26      3
420           doctor-008  2026-03-20      3
463           doctor-009  2026-03-12      3
507           doctor-010  2026-02-23      5
519           doctor-010  2026-03-11      4
572       

In [21]:
for threshold in [3, 4, 5]:
    predicted_anomalous = holdout_df["votes"] >= threshold
    actual_anomalous = holdout_df["label"] == "anomalous"

    tp = (predicted_anomalous & actual_anomalous).sum()
    fp = (predicted_anomalous & ~actual_anomalous).sum()
    fn = (~predicted_anomalous & actual_anomalous).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    print(f"Threshold {threshold}/5 -> TP={tp}, FP={fp}, FN={fn}, Precision={precision:.3f}, Recall={recall:.3f}")

Threshold 3/5 -> TP=28, FP=41, FN=7, Precision=0.406, Recall=0.800
Threshold 4/5 -> TP=26, FP=22, FN=9, Precision=0.542, Recall=0.743
Threshold 5/5 -> TP=24, FP=8, FN=11, Precision=0.750, Recall=0.686


In [22]:
full_df = split_df.copy()

for name, model in models_fitted.items():
    full_df[f"{name}_pred"] = model.predict(X_all_scaled)

full_df["votes"] = full_df[pred_cols].sum(axis=1)
full_df["anomaly_type"] = full_df.set_index(["actorID", "date"]).index.map(anomaly_type_map)

for threshold in [3, 4, 5]:
    predicted_anomalous = full_df["votes"] >= threshold
    actual_anomalous = full_df["label"] == "anomalous"

    tp = (predicted_anomalous & actual_anomalous).sum()
    fp = (predicted_anomalous & ~actual_anomalous).sum()
    fn = (~predicted_anomalous & actual_anomalous).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    print(f"Threshold {threshold}/5 -> TP={tp}, FP={fp}, FN={fn}, Precision={precision:.3f}, Recall={recall:.3f}")

print("\nZero-vote misses (full dataset):")
misses_full = full_df[(full_df["label"] == "anomalous") & (full_df["votes"] == 0)]
print(misses_full["anomaly_type"].value_counts())

Threshold 3/5 -> TP=62, FP=128, FN=19, Precision=0.326, Recall=0.765
Threshold 4/5 -> TP=60, FP=74, FN=21, Precision=0.448, Recall=0.741
Threshold 5/5 -> TP=54, FP=29, FN=27, Precision=0.651, Recall=0.667

Zero-vote misses (full dataset):
anomaly_type
bigram_violation     8
order_composition    6
Name: count, dtype: int64
